<a href="https://colab.research.google.com/github/ciril7/AI-ML-Internship/blob/main/Day%207/Sentimental_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

df=pd.read_csv("/content/IMDB Dataset.csv",encoding='latin1', engine='python', on_bad_lines='skip')

print(df.head())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [2]:
df['sentiment']=df['sentiment'].map({
    'positive':1,
    'negative':0
})

In [3]:
import re

negation_words=[
    "not good","no good","never good","isn't good","aren't good",
    "wasn't good","weren't good","don't feel good","doesn't look good",
    "didn't seem good","won't be good","wouldn't be good","can't be good",
    "couldn't be good","shouldn't be good","hasn't been good","haven't been good",
    "hadn't been good","hardly good","scarcely good","barely good","not very good",
    "not really good","not that good","not so good","not quite good",
    "not particularly good","not entirely good","not always good","not even good",
    "no longer good","far from good","by no means good"
]


def clean_text(text):
  text=text.lower()

  text=re.sub(r"[^a-zA-Z\s']"," ",text)

  for phrase in negation_words:
    text=text.replace(phrase,phrase.replace(" ","_"))

  return text


In [4]:
df['review']=df['review'].apply(clean_text)

In [5]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    df['review'],
    df['sentiment'],
    test_size=0.2,
    random_state=42
)


In [6]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size=20000
max_len=250

tokenizer=Tokenizer(num_words=vocab_size,oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq=tokenizer.texts_to_sequences(X_train)
X_test_seq=tokenizer.texts_to_sequences(X_test)

X_train_pad=pad_sequences(X_train_seq,maxlen=max_len,padding='post')
X_test_pad=pad_sequences(X_test_seq,maxlen=max_len,padding='post')

In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout

model=Sequential([
    Embedding(vocab_size,128,input_length=max_len),

    LSTM(128,dropout=0.3,recurrent_dropout=0.3),

    Dense(64,activation='relu'),
    Dropout(0.3),


    Dense(1,activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [8]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [9]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:
history=model.fit(
    X_train_pad,
    y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=64
)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 432s 853ms/step - accuracy: 0.5504 - loss: 0.6709 - val_accuracy: 0.5761 - val_loss: 0.6493
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 431s 863ms/step - accuracy: 0.6003 - loss: 0.6167 - val_accuracy: 0.6224 - val_loss: 0.6152
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 432s 864ms/step - accuracy: 0.7891 - loss: 0.4955 - val_accuracy: 0.8009 - val_loss: 0.5216
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 422s 844ms/step - accuracy: 0.7889 - loss: 0.4784 - val_accuracy: 0.7232 - val_loss: 0.6366
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 430s 859ms/step - accuracy: 0.8172 - loss: 0.4313 - val_accuracy: 0.7871 - val_loss: 0.4877


In [11]:
loss,acc = model.evaluate(X_test_pad,y_test)
print("Test Accuracy:",acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 33s 103ms/step - accuracy: 0.7848 - loss: 0.4902
Test Accuracy: 0.7847999930381775


In [12]:
def predict_sentiment(review):
  review=clean_text(review)
  seq=tokenizer.texts_to_sequences([review])
  padded=pad_sequences(seq,maxlen=max_len,padding='post')
  prediction=model.predict(padded)[0][0]

  print("\nReview:",review)
  print('Score:',prediction)

  if prediction>=0.5:
    print("Sentiment:Positive 😊")
  else:
    print("Sentiment:Negative 😒")


In [14]:
predict_sentiment("This movie was bad")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step

Review: this movie was bad
Score: 0.34589672
Sentiment:Negative 😒
